# 🛠️ ChaosOps: Training RL Agents for Autonomous Incident Recovery

**GitHub**: [orpheusdark/Chaosops](https://github.com/orpheusdark/Chaosops)

This notebook demonstrates the end-to-end training and evaluation of a PPO-based agent on the ChaosOps-RC environment. ChaosOps simulates a multi-service distributed system with stochastic failure propagation and hidden root causes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/orpheusdark/Chaosops/blob/main/ChaosOps_Training_Colab.ipynb)

In [ ]:
# 1. Setup Environment
!rm -rf Chaosops
!git clone https://github.com/orpheusdark/Chaosops.git
%cd Chaosops
!pip install -r requirements.txt
!pip install matplotlib numpy torch

import sys
import os
sys.path.append('/content/Chaosops')

# Ensure we're in the right directory
if not os.path.exists('evaluation/robustness_eval.py'):
    print("Error: evaluation/robustness_eval.py not found!")
    print("Current directory:", os.getcwd())
    print("Contents:", os.listdir('.'))
else:
    print("Setup complete. Current directory:", os.getcwd())

## 🚀 Run Full Pipeline

The following cell executes the `run_all()` pipeline, which:
1.  Evaluates a **Baseline Agent** (Random Policy).
2.  Trains a **PPO Agent** on the multi-service environment.
3.  Evaluates the **Trained Agent**.
4.  Generates diagnostic plots and performance metrics.

In [ ]:
from train import run_all
run_all()

## 🧪 Robustness Evaluation Suite

Run the new tiered robustness evaluation framework to see how the policy performs under noise, tool failure, structural drift, and worst-case cascades.

In [ ]:
import os
import sys

# Ensure we're in the correct directory for local development
if not os.path.exists('evaluation/robustness_eval.py'):
    # If running locally and not in the project directory, change to it
    if os.path.exists('Chaosops/evaluation/robustness_eval.py'):
        os.chdir('Chaosops')
    elif os.path.exists('../Chaosops/evaluation/robustness_eval.py'):
        os.chdir('../Chaosops')

# Add current directory to Python path
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {os.getcwd() in sys.path}")

In [ ]:
import os
import sys

# Ensure we're in the correct directory for Colab
if '/content/Chaosops' not in sys.path:
    sys.path.append('/content/Chaosops')

# Try importing with different approaches
try:
    from evaluation import RobustnessEvaluator, BaselineAgent
    print("Successfully imported from evaluation package")
except ImportError:
    try:
        from evaluation.robustness_eval import RobustnessEvaluator
        from evaluation.eval_script import BaselineAgent
        print("Successfully imported directly from modules")
    except ImportError as e:
        print(f"Import failed: {e}")
        print("Current directory:", os.getcwd())
        print("Python path:", sys.path)
        print("Files in evaluation directory:", os.listdir('evaluation') if os.path.exists('evaluation') else "evaluation directory not found")
        raise

import json

# Evaluate a baseline agent under the new robustness tiers

evaluator = RobustnessEvaluator(seed=42)
baseline_agent = BaselineAgent()
results = evaluator.evaluate_agent(baseline_agent, num_episodes=20, curriculum_level=2)

print("=== Robustness Evaluation Summary ===")
print(json.dumps(results["summary"], indent=2))
for tier_name, tier_stats in results["tiers"].items():
    print(f"\n--- {tier_name} ---")
    print(f"success_rate: {tier_stats['success_rate']:.1%}")
    print(f"mean_reward: {tier_stats['mean_reward']:.3f}")
    print(f"mean_mttr: {tier_stats['mean_mttr']:.2f}")
    print(f"std_reward: {tier_stats['std_reward']:.3f}")
    print(f"worst_10pct_reward: {tier_stats['worst_10pct_reward']:.3f}")

## 🧠 Advanced: LLM-based SRE with GRPO

Move beyond simple MLPs. This section uses **Hugging Face TRL** to train a **Small Language Model (Qwen2.5-0.5B)** as an on-call agent. 

**GRPO (Group Relative Policy Optimization)** allows the model to reason through multiple diagnostic paths and optimize for the one that resolves the incident fastest with the least amount of noise.

In [ ]:
# Install TRL and dependencies
!pip install trl transformers accelerate -U

# Run the GRPO training script
# Note: This requires a GPU with at least 12GB VRAM (T4/L4 in Colab)
!python train_grpo.py

## 📊 Visualization

Let's look at the training results.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import json

# Display Reward Curve
try:
    img = mpimg.imread('results/reward_curve.png')
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
except:
    print("Reward curve not found. Did the training finish?")

# Display Metrics
try:
    with open('results/metrics.json', 'r') as f:
        metrics = json.load(f)
    print("\n--- PERFORMANCE METRICS ---")
    print(json.dumps(metrics, indent=2))
except:
    print("Metrics JSON not found.")